# Computational Theory: SHA-256 Implementation

## Problem 1: Representing SHA-256 Data

Explain how you would represent:

32-bit words,
sequences of 32-bit words,
input messages,
512-bit message blocks,
the final 256-bit hash value,
big-endian integers.
For each representation, explain why it is appropriate and demonstrate your choices with small Python examples.

### 32-bit word
A 32-bit word is an integer between 0 and 2^32 - 1 (hexadecimal 0x00000000 to 0xFFFFFFFF, 4,294,967,295). Int is the simplest choise because SHA performs bitwise operations on words and ints support those. You can use & operator to truncate int to 32 bit, anything above it. That's how you limit int.

In [2]:
# A 32-bit word constant
w = 0x6A09E667

# Simulating 32-bit addition with overflow truncation
a = 0xFFFFFFFF
b = 0x00000005
result = (a + b) & 0xFFFFFFFF  # Keeps only the lowest 32 bits

print(f"Hex: {hex(result)}")  # Hex: 0x4

Hex: 0x4


### Sequences of 32-bit Words
A series of 32-bit values can be represented by list of ints in python. Why it's appropriate: Lists support fast indexed lookups (words[i]), slice access, and in-place updates, making them intuitive for algorithmic round loops.


In [3]:
# Initial state of 8 working variables (each a 32-bit word)
H = [
    0x6A09E667, 0xBB67AE85, 0x3C6EF372, 0xA54FF53A,
    0x510E527F, 0x9B05688C, 0x1F83D9AB, 0x5BE0CD19
]

# Accessing word 0
print(f"First word: {hex(H[0])}")

First word: 0x6a09e667


### Input Messages
The raw data before padding and hashing (text, files, or binary streams). Use bytes for this, array of bytes. Becasue hashing operates purely on raw binary bytes. So you convert strings into bytes.

In [ ]:
# Converting user text to raw bytes using UTF-8
message_str = "hi, ivan"
message_bytes = message_str.encode("utf-8") 
# OR 
message_bytes_2 = b"hi, ivan"

print(message_bytes)        # b'hi, ivan'
print(list(message_bytes))  # [104, 105, 44, 32, 105, 118, 97, 110]
print(message_bytes_2)        # b'hi, ivan'
print(list(message_bytes_2))  # [104, 105, 44, 32, 105, 118, 97, 110]

b'hi, ivan'
[104, 105, 44, 32, 105, 118, 97, 110]
b'hi, ivan'
[104, 105, 44, 32, 105, 118, 97, 110]


### 512-bit Message Blocks
Hashing algorithms process padded input in chunks of 512 bits (which equals 64 bytes, or 16 thirty-two-bit words). Chosen representation: A bytes object of length 64, or a list of 16 ints unpacked from it. Why it works: SHA-256 takes 64-byte chunks directly from the padded message and immediately slices or unpacks them into 16 individual 32-bit words for processing.

In [6]:
import struct

# A 64-byte (512-bit) chunk
block_bytes = b"A" * 64
print(f"Block byte length: {len(block_bytes)}")  # 64 bytes = 512 bits

# Unpack 64 bytes into sixteen 32-bit big-endian words
# '>16I' means: '>' (big-endian), '16' (count), 'I' (unsigned 32-bit int)
words = list(struct.unpack(">16I", block_bytes))
print(f"Number of words: {len(words)}")  # 16

Block byte length: 64
Number of words: 16


### Final 256-bit Hash Value

The completed digest generated by the algorithm (256 bits = 32 bytes = 8 thirty-two-bit words). Chosen representation: A 32-byte bytes object for binary data, or a 64-character hexadecimal str for human-readable output. Why it's appropriate: bytes is ideal for machine transmission (signatures, storage), while a 64-character hex string is the universal standard for displaying digests to users.

In [7]:
# Example 8 final 32-bit state registers
H_final = [
    0xBA7816BF, 0x8F01CFEA, 0x414140DE, 0x5DAE2223,
    0xB00361A3, 0x96177A9C, 0xB410FF61, 0xF20015AD
]

# Format as 64-character hex string (each word is 8 hex chars, padded with zeros)
hex_digest = "".join(f"{word:08x}" for word in H_final)
print(f"Hex digest ({len(hex_digest)} chars): {hex_digest}")

# Or convert to 32 raw bytes
import struct
raw_digest = struct.pack(">8I", *H_final)
print(f"Raw digest length: {len(raw_digest)} bytes")  # 32 bytes = 256 bits

Hex digest (64 chars): ba7816bf8f01cfea414140de5dae2223b00361a396177a9cb410ff61f20015ad
Raw digest length: 32 bytes


### Big-Endian Integers

Big-endian means the most significant byte (the "biggest" part of the number) is stored first (at the lowest memory address / leftmost position). 1,234 where 1 (a thousand) is the most significant number and 4 is just 4. Chosen representation: Python's built-in int.from_bytes() and int.to_bytes() with byteorder='big', or the struct module using the > format prefix. Why it's appropriate: SHA-256 and network protocols strictly mandate big-endian byte order. Built-in Python methods handle byte packing and unpacking safely without manual bit shifting.

In [ ]:
# 1. Converting 4 bytes to a 32-bit big-endian integer
byte_data = b"\x12\x34\x56\x78"

# 0x12 is the highest byte, 0x78 is the lowest
number = int.from_bytes(byte_data, byteorder="big")
print(f"Integer to hex: {hex(number)}")  # 0x12345678

# 2. Converting an integer back to 4 big-endian bytes
packed_bytes = number.to_bytes(4, byteorder="big")
print(f"Back to bytes: {packed_bytes}")   # b'\x124Vx' (hex 0x12, 0x34, 0x56, 0x78)

## Problem 2: SHA-256 Bitwise Operations



$\text{Ch}(x, y, z)$. Choose. It acts like an array of 32 independent selector switches: for every single bit position, the bit in $x$ determines whether the output takes its bit from $y$ or from $z$.

In [ ]:
import numpy as np

def Ch(x: np.uint32, y: np.uint32, z: np.uint32) -> np.uint32:
    """
    Computes the SHA-256 Choose function: Ch(x, y, z) = (x AND y) XOR (NOT x AND z).
    
    For each bit position:
      - If the bit in x is 1, the output bit is taken from y.
      - If the bit in x is 0, the output bit is taken from z.
      
    Parameters:
        x, y, z : np.uint32 or int
            32-bit unsigned integers.
            
    Returns:
        np.uint32: 32-bit result of the bitwise selection.
    """
    # Enforce 32-bit unsigned integer types
    x = np.uint32(x)
    y = np.uint32(y)
    z = np.uint32(z)
    
    # Compute: (x AND y) XOR ((NOT x) AND z)
    return np.uint32((x & y) ^ (~x & z))

## Problem 3: Generating the SHA-256 Constants


## Problem 4: Padding and Parsing Messages


## Problem 5: The SHA-256 Compression Function


## Problem 6: Complete SHA-256
